Age and Weight

In [5]:
# Add DOB and Avg_Weight_Kg from Sickbay to df_combined
import pandas as pd

# File paths
excel_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_combined.xlsx"
csv_dob_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/DOB.csv"
csv_weight_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/PARDS_Avg_Weight_from_Sickbay.csv"

# Reading files
df_sheet11 = pd.read_excel(excel_file, sheet_name="Sheet11")
print(df_sheet11.shape)
df_dob = pd.read_csv(csv_dob_file)
df_weight = pd.read_csv(csv_weight_file)

# Ensure MRN is a string and 9 digits long
df_sheet11['MRN'] = df_sheet11['MRN'].astype(str).str.zfill(9)
df_dob['MRN'] = df_dob['MRN'].astype(str).str.zfill(9)

# Ensure PID is a string
df_weight['PID'] = df_weight['PID'].astype(str)
df_sheet11['PID'] = df_sheet11['PID'].astype(str)

# Handle duplicates in MRN by keeping the first occurrence
df_dob_unique = df_dob.groupby('MRN', as_index=False).first()

# Map DOB from df_dob_unique to df_sheet11 based on MRN
df_sheet11['DOB'] = df_sheet11['MRN'].map(df_dob_unique.set_index('MRN')['DOB'])

# Directly assign Avg_Weight_Kg based on the order in df_weight
df_sheet11['Avg_Weight_Kg'] = df_weight['Avg_Weight_Kg'].values

# Display the first few rows of the updated dataframe for verification
print("Updated df_sheet11 with DOB and Avg_Weight_Kg:")
print(df_sheet11.shape)
print(df_sheet11.head())


(285, 128)
Updated df_sheet11 with DOB and Avg_Weight_Kg:
(285, 130)
    PID        MRN       Age  Weight  Weight_Pounds  Weight_Kg  \
0  1288  101607118  0.828285  437.04      27.315000  12.389865   
1  1288  101607118  0.828285  437.04      27.315000  12.389865   
2  1293  101666704  0.409113     NaN       0.000000   0.000000   
3  1293  101666704  0.409113     NaN       0.000000   0.000000   
4  1748  101643395  1.277937  444.45      27.778125  12.599935   

       1st_Time_Start       1st_Time_Stop     12th_Time_Start  \
0 2023-12-01 11:47:45 2023-12-01 12:47:45 2023-12-01 23:47:45   
1 2023-11-01 08:36:01 2023-11-01 09:36:01 2023-11-01 20:36:01   
2 2023-08-01 09:03:01 2023-08-01 10:03:01 2023-08-01 21:03:01   
3 2023-09-01 04:26:00 2023-09-01 05:26:00 2023-09-01 16:26:00   
4 2024-02-21 02:44:01 2024-02-21 03:44:01 2024-02-21 14:44:01   

       12th_Time_Stop  ... SD Sub-band 13 SD Sub-band 14 SD Sub-band 15  \
0 2023-12-02 00:47:45  ...       0.059961       0.061845       0.062

In [6]:
# Update Age by using 1st_Time_Start - DOB
import pandas as pd

# Ensure the "1st_Time_Start" and "DOB" columns are in datetime format
df_sheet11['1st_Time_Start'] = pd.to_datetime(df_sheet11['1st_Time_Start'])
df_sheet11['DOB'] = pd.to_datetime(df_sheet11['DOB'])

# Calculate Age_New_Year in years with 2 decimal places
df_sheet11['Age_New_Year'] = (df_sheet11['1st_Time_Start'] - df_sheet11['DOB']).dt.days / 365.25
df_sheet11['Age_New_Year'] = df_sheet11['Age_New_Year'].round(2)

# Display the first few rows of the updated dataframe for verification
print("Updated df_sheet11 with Age_New_Year:")
print(df_sheet11[['DOB', '1st_Time_Start', 'Age_New_Year']].head())
print(df_sheet11.shape)


Updated df_sheet11 with Age_New_Year:
                  DOB      1st_Time_Start  Age_New_Year
0 2022-06-15 11:48:00 2023-12-01 11:47:45          1.46
1 2022-06-15 11:48:00 2023-11-01 08:36:01          1.38
2 2022-09-21 00:00:00 2023-08-01 09:03:01          0.86
3 2022-09-21 00:00:00 2023-09-01 04:26:00          0.94
4 2022-06-14 00:00:00 2024-02-21 02:44:01          1.69
(285, 131)


In [7]:
from openpyxl import load_workbook

# File path
output_file = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_combined.xlsx"

# Load the existing workbook and append the updated DataFrame as a new sheet
with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
    # Save df_sheet11 as a new sheet named "Sheet12"
    df_sheet11.to_excel(writer, sheet_name="Sheet12", index=False)

print("Updated df_combined.xlsx with Sheet12 successfully!")


Updated df_combined.xlsx with Sheet12 successfully!


### Calculate the Tidal Volume: TV (mL/kg) = eVT (mL) / Weight (kg)

In [1]:
# Demo (for individual file)
import os
import pandas as pd

# Define paths
data_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/01_Vital_Raw_282_1st/OSI_285_1st/TWs_284_1st/PEEP_Settings_284_1st/Abnormal_Detection_282_1st"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_combined.xlsx"
sheet_to_read = "Sheet17"

# Read the Excel file
excel_data = pd.ExcelFile(excel_path)
df_combined = excel_data.parse(sheet_to_read)

# Process each CSV file
csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]
if csv_files:
    first_file = csv_files[0]
    print(first_file)
    first_file_path = os.path.join(data_dir, first_file)

    # Read the CSV file
    df_csv = pd.read_csv(first_file_path)

    # Match the file name with the "FileName_Vital_1st" column in df_combined
    matching_row = df_combined[df_combined["FileName_Vital_1st"] == first_file]

    if not matching_row.empty:
        # Identify the column for eVT (first non-empty column among "CDGR - eVT", "AVEA - eVT", "SVU - eVT")
        eVT_columns = ["CDGR - eVT", "AVEA - eVT", "SVU - eVT"]
        eVT_column = next((col for col in eVT_columns if col in df_csv.columns and not df_csv[col].isnull().all()), None)

        if eVT_column:
            # Average the eVT column
            avg_eVT = df_csv[eVT_column].mean()
            print(avg_eVT)

            # std the eVT column
            std_eVT = df_csv[eVT_column].std()
            print(std_eVT)

            # Calculate TV(mL/Kg)
            avg_weight_kg = matching_row["Avg_Weight_Kg"].iloc[0]
            print(avg_weight_kg)

            tv_ml_per_kg_mean = avg_eVT / avg_weight_kg
            print(tv_ml_per_kg_mean)

            tv_ml_per_kg_std = std_eVT / avg_weight_kg
            print(tv_ml_per_kg_std)


802274_20230901_06_Vital_1st.csv
34.69019778715733
8.16473286528344
4.8
7.227124538991111
1.7009860136007169


In [2]:
# For all 282 files:
import os
import pandas as pd

# Define paths
data_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/VitalData/01_Vital_Raw_282_1st/OSI_285_1st/TWs_284_1st/PEEP_Settings_284_1st/Abnormal_Detection_282_1st"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/df_combined.xlsx"
sheet_to_read = "Sheet17"
sheet_to_save = "Sheet18"

# Read the Excel file
excel_data = pd.ExcelFile(excel_path)
df_combined = excel_data.parse(sheet_to_read)

# Create a copy of the dataframe for Sheet18
df_sheet18 = df_combined.copy()

# Process each CSV file
for file in os.listdir(data_dir):
    if file.endswith(".csv"):
        file_path = os.path.join(data_dir, file)

        # Read the CSV file
        df_csv = pd.read_csv(file_path)

        # Match the file name with the "FileName_Vital_1st" column in df_combined
        matching_row = df_combined[df_combined["FileName_Vital_1st"] == file]

        if not matching_row.empty:
            # Identify the column for eVT (first non-empty column among "CDGR - eVT", "AVEA - eVT", "SVU - eVT")
            eVT_columns = ["CDGR - eVT", "AVEA - eVT", "SVU - eVT"]
            eVT_column = next((col for col in eVT_columns if col in df_csv.columns and not df_csv[col].isnull().all()), None)

            if eVT_column:
                # Average the eVT column
                avg_eVT = df_csv[eVT_column].mean()
                # std the eVT column
                std_eVT = df_csv[eVT_column].std()

                # Calculate TV(mL/Kg)
                avg_weight_kg = matching_row["Avg_Weight_Kg"].iloc[0]
                tv_ml_per_kg_mean = avg_eVT / avg_weight_kg
                tv_ml_per_kg_std = std_eVT / avg_weight_kg

                # Add the results to the new columns in Sheet18
                df_sheet18.loc[matching_row.index, "avg_eVT"] = round(avg_eVT, 2)
                df_sheet18.loc[matching_row.index, "std_eVT"] = round(std_eVT, 2)
                df_sheet18.loc[matching_row.index, "TV_mean(mL/Kg)"] = round(tv_ml_per_kg_mean, 2)
                df_sheet18.loc[matching_row.index, "TV_std(mL/Kg)"] = round(tv_ml_per_kg_std, 2)

# Save the updated dataframe to Sheet18
with pd.ExcelWriter(excel_path, mode='a', if_sheet_exists='replace') as writer:
    df_sheet18.to_excel(writer, sheet_name=sheet_to_save, index=False)

print("Processing complete. Updated data saved to Sheet18.")


Processing complete. Updated data saved to Sheet18.
